# Research Proposal: Modeling Daily Parking Ticket Counts in Milwaukee

### John Eckberg

## Overview
[Milwaukee plans 65,000 more parking tickets, longer meter hours to fill transportation budget gap](https://www.jsonline.com/story/news/local/milwaukee/2025/10/21/milwaukee-aims-for-550k-parking-tickets-longer-meter-hours-in-2026/86817755007/) The city of Milwaukee relies on parking ticket revenue in order to fill budget gaps, a pressing issue given the [current deficit](https://urbanmilwaukee.com/2025/03/12/mke-county-county-faces-47-million-2026-budget-deficit/) the county is facing.
This project plans to examine how daily parking ticket activity in Milwaukee varies across time and environmental conditions. Using parking citation records obtained through a Freedom of Information Act request (2014–2022), I propose to model daily ticket counts and evaluate whether temporal, weather, and holiday-related factors explain variation in enforcement volume.

## Research Question
What is the daily rate of parking tickets issued in Milwaukee, and which factors most strongly influence that rate (if any)? In particular, I propose to test whether day of week, month/seasonality, holidays, and precipitation are associated with higher or lower daily ticket counts, including by citation type.

## Hypothesis
- Null Hypothesis: day of week, month/seasonality, holidays, and precipitation do not affect daily ticket counts.
- Alternative Hypothesis: at least one temporal or environmental factor affects daily ticket counts.

## Data Sources
- **Parking citations:** Milwaukee parking ticket records, initially loaded for 2014, with fields including `ISSUENO`, `ISSUEDATE`, `LOCATIONDESC1`, and `VIODESCRIPTION`.
- **Holiday data:** U.S. holiday dates (2004–2021) pulled from Kaggle to identify atypical traffic/parking demand days.
- **Weather data:** Open-Meteo daily weather for Milwaukee, including maximum temperature, minimum temperature, precipitation, and snowfall.


## Method
The analysis would likely focus on a geographically meaningful subset of Milwaukee (e.g., downtown, Lower East Side, Deer District). Because the response variable is a daily count, I propose that the initial model choice would be a Poisson regression GLM.

### Currently Planned Features
- **Calendar effects:** day of week, month, weekend indicator, and potential seasonal terms.
- **Cyclical time encoding:** sine/cosine transforms for month and day-of-week to preserve circular structure.
- **Holiday effects:** binary `is_holiday` indicator, one-hot encoding for major holiday names.
- **Weather exposure:** `max_temp`, `min_temp`, `precipitation`, and `snowfall`; optional threshold features (e.g., heavy precipitation day).
- **Autocorrelation:** lagged ticket count features (e.g., 1-day and 7-day lags) and short rolling averages.



In [ ]:
import pandas as pd
import requests
import kagglehub 
import openpyxl
from pathlib import Path

c:\Users\mathewsj\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Loading in the parking data [All Years now]  --  Runs in about 10 mins

#tickets_df = pd.read_excel("data/tickets_csv/2014_parking_citations_issued_open_records_request.xlsx")
data_dir = Path("data/tickets_csv")

tickets_dfs = []

for file in sorted(data_dir.glob("*.xlsx")):
    df = pd.read_excel(file)
    tickets_dfs.append(df)

tickets = pd.concat(tickets_dfs, ignore_index=True)

tickets.sample(5)

,ISSUENO,ISSUEDATE,LOCATIONDESC1,VIODESCRIPTION
425369,479393751,2014-09-07,1414 N ASTOR ST,PARKED WITHIN 4 FEET OF DRIVE OR ALLEY
1258531,489027965,2016-01-14,2937 N 40TH ST,NIGHT PARKING
384100,479010560,2014-08-15,425 W MICHIGAN ST,METER PARKING VIOLATION
3844559,121253650,2020-10-27,310 E ERIE ST,METER PARKING VIOLATION
275094,477375732,2014-06-10,3304 N SHEPARD AVE,RESIDENTIAL PARKING PROGRAM


In [37]:
tickets.info()

<class 'pandas.DataFrame'>
RangeIndex: 4865096 entries, 0 to 4865095
Data columns (total 4 columns):
 #   Column          Dtype         
---  ------          -----         
 0   ISSUENO         int64         
 1   ISSUEDATE       datetime64[us]
 2   LOCATIONDESC1   object        
 3   VIODESCRIPTION  str           
dtypes: datetime64[us](1), int64(1), object(1), str(1)
memory usage: 148.5+ MB


In [3]:
# load in holiday data (https://www.kaggle.com/datasets/donnetew/us-holiday-dates-2004-2021)

path = kagglehub.dataset_download("donnetew/us-holiday-dates-2004-2021")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\mathewsj\.cache\kagglehub\datasets\donnetew\us-holiday-dates-2004-2021\versions\1


In [38]:
#holidays_df = pd.read_csv("C:/Users/eckbergj/.vscode/Milwaukee-Ticket-Data-Analysis/data/US Holiday Dates (2004-2021).csv")
holidays_df = pd.read_csv("data/US Holiday Dates (2004-2021).csv")

holidays_df.head(10)

,Date,Holiday,WeekDay,Month,Day,Year
0,2004-07-04,4th of July,Sunday,7,4,2004
1,2005-07-04,4th of July,Monday,7,4,2005
2,2006-07-04,4th of July,Tuesday,7,4,2006
3,2007-07-04,4th of July,Wednesday,7,4,2007
4,2008-07-04,4th of July,Friday,7,4,2008
5,2009-07-04,4th of July,Saturday,7,4,2009
6,2010-07-04,4th of July,Sunday,7,4,2010
7,2011-07-04,4th of July,Monday,7,4,2011
8,2012-07-04,4th of July,Wednesday,7,4,2012
9,2013-07-04,4th of July,Thursday,7,4,2013


In [ ]:
# weather data (https://open-meteo.com/)  [extended from 2012 - 2022]
test_data = requests.get("https://archive-api.open-meteo.com/v1/archive?latitude=43.0389&longitude=-87.9065&start_date=2014-01-01&end_date=2022-12-31&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum&timezone=America%2FChicago")
test_data_json = test_data.json()

# minor cleaning of the weather data to get it into a dataframe
weather_df = pd.DataFrame(test_data_json['daily'])
weather_df['time'] = pd.to_datetime(weather_df['time'])
weather_df = weather_df.rename(columns={'time': 'date', 'temperature_2m_max': 'max_temp', 'temperature_2m_min': 'min_temp', 'precipitation_sum': 'precipitation','snowfall_sum': 'snowfall'})
weather_df.sample(10)


,date,max_temp,min_temp,precipitation,snowfall
1116,2017-01-21,11.5,2.2,0.6,0.00
2747,2021-07-10,21.9,14.6,0.0,0.00
1126,2017-01-31,2.1,-2.4,2.3,1.47
2951,2022-01-30,-0.3,-9.2,0.0,0.00
2964,2022-02-12,-7.4,-12.8,0.0,0.00
3264,2022-12-09,2.0,0.9,10.4,6.79
1068,2016-12-04,2.1,0.4,9.1,5.88
1181,2017-03-27,6.5,3.2,0.0,0.00
2417,2020-08-14,27.8,18.0,0.0,0.00
2540,2020-12-15,-1.6,-12.2,0.0,0.00


In [45]:
# clean and align date keys before joining
tickets_clean = tickets.copy()
holidays_clean = holidays_df.copy()
weather_clean = weather_df.copy()

# Datetime object
tickets_clean['join_date'] = pd.to_datetime(tickets_clean['ISSUEDATE'], errors='coerce').dt.normalize()
holidays_clean['join_date'] = pd.to_datetime(holidays_clean['Date'], errors='coerce').dt.normalize()
weather_clean['join_date'] = pd.to_datetime(weather_clean['date'], errors='coerce').dt.normalize()


# merge tickets with holidays and weather on a consistent datetime key
combined_tickets_and_holidays_df = pd.merge(
    tickets_clean,
    holidays_clean,
    how='left',      # SWITCHED from 'inner' to 'left' to keep all tickets, even those without a holiday match (-> 'NAN')
    on='join_date'
 )

combined_all_df = pd.merge(
    combined_tickets_and_holidays_df,
    weather_clean,
    how='left',
    on='join_date',
    suffixes=('_holiday', '_weather')
 )

# Dropping REDUNDANT columns from the joins and the Holiday data (since many become NAN, will grab the same date info later on)
combined_all_df = combined_all_df.drop(columns=['ISSUEDATE','Date','date','WeekDay', 'Month', 'Day', 'Year'])

# sample of the combined dataframe
combined_all_df[combined_all_df['Holiday'].notnull()].sample(5)
#combined_all_df.sample(5)

,ISSUENO,LOCATIONDESC1,VIODESCRIPTION,join_date,Holiday,max_temp,min_temp,precipitation,snowfall
1791074,494574430,3224 S 16TH ST,PARKED IN EXCESS OF 1 HOUR PROHIBITED,2016-11-23,Thanksgiving Eve,6.5,4.2,7.5,0.42
3426981,116923645,2161 S ROBINSON AV,NIGHT PARKING - WRONG SIDE,2019-08-31,Labor Day Weekend,21.0,12.4,0.2,0.00
493850,480363693,2652 S 9TH PL,NIGHT PARKING,2014-10-13,Columbus Day,16.4,11.7,29.2,0.00
2361893,106148011,1515 E KANE PL,PARKED IN EXCESS OF 2 HOURS PROHIBITED,2017-11-11,Veterans Day,1.5,-2.8,0.0,0.00
2433020,106750346,2809 W HIGHLAND BL,IMPROPERLY DISPLAYED VEHICLE REGISTRATION,2018-01-01,New Year's Day,-15.3,-22.7,0.0,0.00


### Feature Engineering

In [47]:
# Use Pandas DateTime to get WeekDay, Day (number), Month, Year from each ticket's join_date (ISSUEDATE)
combined_all_df['WeekDay'] = pd.to_datetime(combined_all_df['join_date'], errors='coerce').dt.day_name()
combined_all_df['Day'] = pd.to_datetime(combined_all_df['join_date'], errors='coerce').dt.day
combined_all_df['Month'] = pd.to_datetime(combined_all_df['join_date'], errors='coerce').dt.month
combined_all_df['Year'] = pd.to_datetime(combined_all_df['join_date'], errors='coerce').dt.year

combined_all_df.sample(5)

,ISSUENO,LOCATIONDESC1,VIODESCRIPTION,join_date,Holiday,max_temp,min_temp,precipitation,snowfall,WeekDay,Day,Month,Year
4741772,130850613,733 N 12TH ST,METER PARKING VIOLATION,2022-09-15,NaN,25.8,13.6,0.0,0.0,Thursday,15,9,2022
3577328,118443054,6227 W MEDFORD AV,FAILURE TO DISPLAY CURRENT REGISTRATION,2019-12-04,NaN,3.6,-0.9,0.0,0.0,Wednesday,4,12,2019
2147150,103776024,2905 N 47TH ST,TOW-AWAY ZONE (BLOCKING TRAFFIC),2017-06-19,Juneteenth,22.5,16.2,5.4,0.0,Monday,19,6,2017
4405939,127222631,6301 W PORT AV,UNREGISTERED/ IMPROPERLY REGISTERED VEHICLE,2022-01-12,NaN,5.6,-1.7,0.0,0.0,Wednesday,12,1,2022
3380663,116414233,3155 N 82ND ST,NIGHT PARKING,2019-08-01,NaN,23.8,11.2,0.0,0.0,Thursday,1,8,2019


In [48]:
# See what Violation categories can be made out of the data
combined_all_df.value_counts('VIODESCRIPTION')

VIODESCRIPTION
NIGHT PARKING                                1527527
METER PARKING VIOLATION                       810877
NIGHT PARKING - WRONG SIDE                    748186
PARKING PROHIBITED BY OFFICIAL SIGN           259562
IMPROPERLY DISPLAYED VEHICLE REGISTRATION     203366
                                              ...   
TESTING IVR                                        2
TEST VIOLATION                                     1
PARKED IN RECREATIONAL AREA                        1
PARKED IN EXCESS OF 24 HOURS PROHIBITED            1
EXCEED 10-HOUR LIMIT AT METER                      1
Name: count, Length: 90, dtype: int64

In [49]:
# Feature for  Winter Storm? Inclement Weather?  Threshold can be adjusted 
combined_all_df['is_snow_emergency'] = ((combined_all_df['VIODESCRIPTION'].str.lower().str.contains('snow|emergency')) |      # in-line OR
                                        (combined_all_df['snowfall'] > 3.0)).astype(int)

# Heavy Rain?
combined_all_df['is_heavy_rain'] = (combined_all_df['precipitation'] > 2.0).astype(int)

# Meter Violation?
combined_all_df['is_meter_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('meter').astype(int) # 1 = Meter Violation, 0 = Not a Meter Violation

# Night Violation?
combined_all_df['is_night_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('night').astype(int) # 1 = Night Violation, 0 = Not a Night Violation

# Registration Violation?
combined_all_df['is_registration_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('registration|unregistered').astype(int) # 1 = Registration Violation, 0 = Not a Registration Violation

# Time Violation ?
combined_all_df['is_time_ticket'] = combined_all_df['VIODESCRIPTION'].str.lower().str.contains('excess|hours|time').astype(int) # 1 = Time Violation, 0 = Not a Time Violation

# Weekend Violation?  Could add Friday as well if wanted
combined_all_df['is_weekend_ticket'] = combined_all_df['WeekDay'].str.lower().str.contains('saturday|sunday').astype(int) # 1 = Weekend Violation, 0 = Not a Weekend Violation


# Create a binary holiday flag immediately after the merge
# If 'Holiday' is not null, it becomes 1. If it is null, it becomes 0.
combined_all_df['is_holiday'] = combined_all_df['Holiday'].notnull().astype(int)

combined_all_df[combined_all_df['is_holiday'] == 1].sample(8)

,ISSUENO,LOCATIONDESC1,VIODESCRIPTION,join_date,Holiday,max_temp,min_temp,precipitation,snowfall,WeekDay,...,Month,Year,is_snow_emergency,is_heavy_rain,is_meter_ticket,is_night_ticket,is_registration_ticket,is_time_ticket,is_weekend_ticket,is_holiday
4389126,127042882,624 W HISTORIC MITCHELL ST,METER PARKING VIOLATION,2021-12-24,Christmas Eve,8.3,-3.5,0.2,0.00,Friday,...,12,2021,0,0,1,0,0,0,0,1
1117125,487643800,322 W MINERAL ST,NIGHT PARKING - WRONG SIDE,2015-10-12,Columbus Day,19.9,12.5,0.0,0.00,Monday,...,10,2015,0,0,0,1,0,0,0,1
4395687,127085265,2706 N FREDERICK AV,PARKED WITHIN 10 FEET OF FIRE HYDRANT,2021-12-31,New Year’s Eve,2.6,-1.2,0.9,0.35,Friday,...,12,2021,0,0,0,0,0,0,0,1
4388938,127107805,1008 W GREENFIELD AV,NIGHT PARKING - WINTER RESTRICTED,2021-12-24,Christmas Eve,8.3,-3.5,0.2,0.00,Friday,...,12,2021,0,0,0,1,0,0,0,1
1933114,101703453,3541 N 11TH ST,NIGHT PARKING,2017-02-14,Valentine’s Day,9.2,0.2,0.1,0.00,Tuesday,...,2,2017,0,0,0,1,0,0,0,1
625393,481746790,704 E WELLS ST,METER PARKING VIOLATION,2014-12-24,Christmas Eve,3.2,1.6,1.4,0.00,Wednesday,...,12,2014,0,0,1,0,0,0,0,1
3498210,117541524,669 N PLANKINTON AV,METER PARKING VIOLATION,2019-10-14,Columbus Day,10.8,1.5,0.0,0.00,Monday,...,10,2019,0,0,1,0,0,0,0,1
3049598,113106545,4313 W DOUGLAS AV,NIGHT PARKING,2018-12-24,Christmas Eve,-0.6,-5.4,0.0,0.00,Monday,...,12,2018,0,0,0,1,0,0,0,1


In [ ]:
# Compressing into a Daily Format: Row per Day, with Ticket Counts and Weather/Holiday Features to be able to then predict the Daily Rate of parking tickets (and the features that affect that)
daily_summary_df = combined_all_df.groupby('join_date').agg(
    # Target Variable: COUNT of the total tickets issued that day
    total_tickets=('ISSUENO', 'count'),

    # Dataset Weather Features:
    max_temp = ('max_temp', 'max'),
    min_temp = ('min_temp', 'min'),
    precipitation = ('precipitation', 'max'),
    snowfall = ('snowfall', 'max'),
    is_snow_emergency = ('is_snow_emergency', 'max'),
    is_heavy_rain = ('is_heavy_rain', 'max'),

    # Calendar Features:
    WeekDay = ('WeekDay', 'first'),
    Year = ('Year', 'first'),
    Month = ('Month', 'first'),
    Day = ('Day', 'first'),
    is_weekend = ('is_weekend_ticket', 'max'),
    is_holiday = ('is_holiday', 'max'),

    # Count up the number of each Binary Feature type per day!
    meter_tickets_count = ('is_meter_ticket', 'sum'),
    night_tickets_count = ('is_night_ticket', 'sum'),
    registration_tickets_count = ('is_registration_ticket', 'sum'),
    time_tickets_count = ('is_time_ticket', 'sum')
).reset_index()

daily_summary_df.sample(10)

,join_date,total_tickets,max_temp,min_temp,precipitation,snowfall,is_snow_emergency,is_heavy_rain,WeekDay,Year,Month,Day,is_weekend,is_holiday,meter_tickets_count,night_tickets_count,registration_tickets_count,time_tickets_count
2743,2021-07-06,295,34.0,17.0,1.8,0.00,0,0,Tuesday,2021,7,6,0,0,91,0,92,14
2749,2021-07-12,608,21.3,16.7,7.7,0.00,0,1,Monday,2021,7,12,0,0,72,321,68,16
1410,2017-11-11,1144,1.5,-2.8,0.0,0.00,0,0,Saturday,2017,11,11,1,1,25,879,84,16
3225,2022-10-31,720,16.4,8.8,0.9,0.00,0,0,Monday,2022,10,31,0,0,97,403,43,69
135,2014-05-16,1873,5.7,2.8,0.0,0.00,0,0,Friday,2014,5,16,0,0,573,760,84,207
1981,2019-06-05,2043,25.2,11.4,2.8,0.00,0,1,Wednesday,2019,6,5,0,0,353,1117,284,12
58,2014-02-28,2133,-6.4,-18.5,0.4,0.35,0,0,Friday,2014,2,28,0,0,567,949,60,239
3062,2022-05-21,1208,18.1,9.0,0.0,0.00,0,0,Saturday,2022,5,21,1,0,159,767,74,20
29,2014-01-30,2388,-0.9,-8.0,0.8,0.63,0,0,Thursday,2014,1,30,0,0,508,1193,87,223
3263,2022-12-08,2137,2.4,-0.1,0.0,0.00,0,0,Thursday,2022,12,8,0,0,256,1194,91,202
